# Instacart: limpieza y estandarización de datos



**Evidencia:** GA1-220501094-01-AA1-EV06  

**Lección 2:** Limpieza y estandarización  



Este notebook desarrolla un pipeline reproducible de limpieza con Pandas, genera un informe de cambios y resuelve siete ejercicios prácticos. Se conservan los datos originales en objetos separados y no se imputan los valores ausentes de `days_since_prior_order`, porque son esperables para el primer pedido de cada usuario.



> Los CSV incluidos en este repositorio son datos pequeños de ejemplo. Las conclusiones y conteos de esta ejecución no representan el conjunto completo de Instacart.

## 1. Preparar el entorno y cargar los datos



El bloque localiza la raíz del proyecto tanto si Jupyter se inicia desde `instacart-analisis/` como desde `instacart-analisis/notebooks/`. Los cinco CSV usan punto y coma como separador.

In [52]:
from pathlib import Path



import pandas as pd

from IPython.display import display



EXPECTED_FILES = (

    "instacart_orders.csv",

    "order_products.csv",

    "products.csv",

    "aisles.csv",

    "departments.csv",

)



candidate_roots = (Path.cwd(), Path.cwd().parent)

PROJECT_ROOT = next(

    (

        candidate.resolve()

        for candidate in candidate_roots

        if all((candidate / "data" / filename).is_file() for filename in EXPECTED_FILES)

    ),

    None,

)



if PROJECT_ROOT is None:

    raise FileNotFoundError(

        "No se encontraron los cinco CSV en data/. Inicia Jupyter desde "

        "instacart-analisis/ o desde instacart-analisis/notebooks/."

    )



DATA_DIR = PROJECT_ROOT / "data"

print(f"Raíz del proyecto: {PROJECT_ROOT}")

print(f"Directorio de datos: {DATA_DIR}")

Raíz del proyecto: D:\TRABAJOS SENA\CIENCIAS DE DATOS\FUNDAMENTOS EN PANDAS\instacart-analisis
Directorio de datos: D:\TRABAJOS SENA\CIENCIAS DE DATOS\FUNDAMENTOS EN PANDAS\instacart-analisis\data


In [53]:
raw_data = {

    Path(filename).stem: pd.read_csv(DATA_DIR / filename, sep=";")

    for filename in EXPECTED_FILES

}



orders_raw = raw_data["instacart_orders"]

order_products_raw = raw_data["order_products"]

products_raw = raw_data["products"]

aisles_raw = raw_data["aisles"]

departments_raw = raw_data["departments"]



print("Archivos cargados:", ", ".join(raw_data))

Archivos cargados: instacart_orders, order_products, products, aisles, departments


## 2. Diagnóstico inicial



Antes de modificar los datos, se revisan dimensiones, tipos, valores ausentes y duplicados. Esta línea base permite explicar qué cambió en el pipeline.

In [54]:
initial_structure = pd.DataFrame(

    [

        {

            "tabla": name,

            "filas": len(frame),

            "columnas": frame.shape[1],

            "valores_ausentes": int(frame.isna().sum().sum()),

            "filas_duplicadas_exactas": int(frame.duplicated().sum()),

        }

        for name, frame in raw_data.items()

    ]

)

display(initial_structure)



for name, frame in raw_data.items():

    print(f"\n{name}: {frame.shape}")

    display(frame.dtypes.rename("tipo").to_frame())

,tabla,filas,columnas,valores_ausentes,filas_duplicadas_exactas
0,instacart_orders,4,7,3,0
1,order_products,4,4,0,0
2,products,3,4,0,0
3,aisles,3,2,0,0
4,departments,2,2,0,0



instacart_orders: (4, 7)


,tipo
order_id,int64
user_id,int64
eval_set,str
order_number,int64
order_dow,int64
order_hour_of_day,int64
days_since_prior_order,float64



order_products: (4, 4)


,tipo
order_id,int64
product_id,int64
add_to_cart_order,int64
reordered,int64



products: (3, 4)


,tipo
product_id,int64
product_name,str
aisle_id,int64
department_id,int64



aisles: (3, 2)


,tipo
aisle_id,int64
aisle,str



departments: (2, 2)


,tipo
department_id,int64
department,str


## 3. Laboratorio guiado 2: pipeline de limpieza con informe de cambios

El laboratorio recarga los archivos originales, registra una fotografia de cada paso (filas, nulos y memoria), aplica funciones pequenas y encadena la limpieza con DataFrame.pipe. Los datos crudos se mantienen intactos para poder comparar los resultados y repetir el proceso.

In [55]:
registro = []


def anotar(df, tabla, paso):
    registro.append(
        {
            "tabla": tabla,
            "paso": paso,
            "filas": len(df),
            "nulos": int(df.isna().sum().sum()),
            "memoria_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
        }
    )
    return df


def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)


def tipos_orders(df):
    return df.astype(
        {
            "order_id": "int32",
            "user_id": "int32",
            "order_number": "int16",
            "order_dow": "int8",
            "order_hour_of_day": "int8",
            "days_since_prior_order": "Int16",
        }
    )


def limpiar_nombres(df):
    df = df.copy()
    df["product_name"] = (
        df["product_name"]
        .fillna("Unknown")
        .str.replace("\xa0", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.title()
    )
    return df


def tipos_order_prods(df):
    return df.astype(
        {
            "order_id": "int32",
            "product_id": "int32",
            "add_to_cart_order": "Int16",
            "reordered": "int8",
        }
    )

In [56]:
orders_crudo = pd.read_csv(DATA_DIR / "instacart_orders.csv", sep=";")
products_crudo = pd.read_csv(DATA_DIR / "products.csv", sep=";")
op_crudo = pd.read_csv(DATA_DIR / "order_products.csv", sep=";")

print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(4, 7) (3, 4) (4, 4)


In [57]:
orders_limpio = (
    orders_crudo.pipe(anotar, "orders", "0 crudo")
    .pipe(quitar_duplicados)
    .pipe(anotar, "orders", "1 sin duplicados")
    .pipe(tipos_orders)
    .pipe(anotar, "orders", "2 tipos")
)

products_limpio = (
    products_crudo.pipe(anotar, "products", "0 crudo")
    .pipe(limpiar_nombres)
    .pipe(anotar, "products", "1 nombres")
)

op_limpio = (
    op_crudo.pipe(anotar, "order_prods", "0 crudo")
    .pipe(tipos_order_prods)
    .pipe(anotar, "order_prods", "1 tipos")
)

orders_clean = orders_limpio
products_clean = products_limpio
order_products_clean = op_limpio
order_prods = order_products_clean
aisles_clean = aisles_raw.copy()
departments_clean = departments_raw.copy()

In [58]:
cleaning_report = pd.DataFrame(registro)
display(cleaning_report)

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,4,3,0.0
1,orders,1 sin duplicados,4,3,0.0
2,orders,2 tipos,4,3,0.0
3,products,0 crudo,3,0,0.0
4,products,1 nombres,3,0,0.0
5,order_prods,0 crudo,4,0,0.0
6,order_prods,1 tipos,4,0,0.0


### Lectura del informe

El informe conserva el estado de cada tabla en cada paso, asi se puede ver si la limpieza cambio filas, nulos o memoria. Los nombres de producto que faltan se sustituyen por "Unknown" para conservar las filas de articulos; los nulos de add_to_cart_order se investigan en el Reto 2, no se imputan sin conocer su causa. Nombres de producto iguales no implican el mismo producto: distintos product_id se conservan para no perder referencias de pedidos.

In [59]:
def validar(orders, products, order_prods):
    assert orders["order_id"].is_unique, "order_id repetido"
    assert orders["order_hour_of_day"].between(0, 23).all(), "hora fuera de rango"
    assert products["product_name"].notna().all(), "productos sin nombre"
    assert not products["product_name"].str.contains(
        "\xa0", regex=False
    ).any(), "espacios invisibles"
    assert order_prods["reordered"].isin([0, 1]).all(), "reordered con valores raros"
    return "Validacion superada"


validation_result = validar(orders_clean, products_clean, order_products_clean)
assert orders_crudo.equals(orders_raw), "La tabla cruda de pedidos fue modificada"
assert products_crudo.equals(products_raw), "La tabla cruda de productos fue modificada"
print(validation_result)
print("Las copias de datos crudos permanecen intactas.")

Validacion superada
Las copias de datos crudos permanecen intactas.


## 4. Reto 2: siete ejercicios de limpieza



Los ejercicios siguientes practican detección, estandarización y validación usando los cinco DataFrames limpiados. Las soluciones muestran resultados verificables y conservan la trazabilidad de los cambios.

### Ejercicio 1. Revisar duplicados y valores de reordered

Comprueba si order_prods tiene filas duplicadas exactas, si se repite algun par order_id + product_id y si reordered solo contiene 0 y 1.

In [60]:
exact_duplicates = int(order_prods.duplicated().sum())
duplicate_pairs = order_prods.duplicated(
    subset=["order_id", "product_id"], keep=False
)
duplicate_pair_count = order_prods.loc[
    duplicate_pairs, ["order_id", "product_id"]
].drop_duplicates().shape[0]
reordered_values = sorted(order_prods["reordered"].dropna().unique().tolist())
reordered_is_binary = order_prods["reordered"].isin([0, 1]).all()

display(
    pd.DataFrame(
        {
            "resultado": [
                exact_duplicates,
                duplicate_pair_count,
                reordered_values,
                reordered_is_binary,
            ]
        },
        index=[
            "filas duplicadas exactas",
            "pares order_id-product_id repetidos",
            "valores de reordered",
            "reordered contiene solo 0 y 1",
        ],
    )
)

,resultado
filas duplicadas exactas,0
pares order_id-product_id repetidos,0
valores de reordered,"[0, 1]"
reordered contiene solo 0 y 1,True


**Interpretacion:** una repeticion de order_id + product_id senalaria el mismo producto repetido dentro de un pedido; valores de reordered distintos de 0 y 1 requieren investigar antes de analizarlos.

### Ejercicio 2. Revisar nombres de pasillos y departamentos

Comprueba si aisles y departments tienen nombres duplicados o con espacios al inicio o al final. Decide si necesitan limpieza de texto.

In [61]:
text_quality_rows = []
for table_name, frame, name_column in [
    ("aisles", aisles_raw, "aisle"),
    ("departments", departments_raw, "department"),
]:
    names = frame[name_column].astype("string")
    text_quality_rows.append(
        {
            "tabla": table_name,
            "nombres_duplicados": int(names.dropna().duplicated().sum()),
            "nombres_con_espacios_laterales": int(
                (names.notna() & names.ne(names.str.strip())).sum()
            ),
        }
    )

text_quality_report = pd.DataFrame(text_quality_rows)
display(text_quality_report)

,tabla,nombres_duplicados,nombres_con_espacios_laterales
0,aisles,1,0
1,departments,0,0


**Interpretacion:** si los conteos de espacios laterales son cero, ese defecto no requiere correccion en estas tablas. Los nombres duplicados se evaluan segun su funcion: varios ids pueden compartir una etiqueta.

### Ejercicio 3. Investigar los nulos de add_to_cart_order

El material identifica 836 nulos. Determina en cuantos pedidos ocurren y cuantos productos tiene, como minimo, cada pedido afectado. No elimines esas filas sin una causa comprobada.

In [62]:
missing_cart_rows = order_prods.loc[order_prods["add_to_cart_order"].isna()]
missing_cart_order_ids = missing_cart_rows["order_id"].drop_duplicates()
products_per_order = order_prods.groupby("order_id").size()
missing_cart_order_sizes = (
    products_per_order.loc[products_per_order.index.isin(missing_cart_order_ids)]
    .sort_values()
    .rename("productos_en_pedido")
    .rename_axis("order_id")
    .reset_index()
)

print(f"Filas con add_to_cart_order nulo: {len(missing_cart_rows):,}")
print(f"Pedidos afectados: {len(missing_cart_order_ids):,}")
if missing_cart_order_sizes.empty:
    print("Esta muestra no contiene pedidos con posicion de carrito nula.")
else:
    print(
        "Minimo de productos por pedido afectado:",
        int(missing_cart_order_sizes["productos_en_pedido"].min()),
    )
    display(missing_cart_order_sizes)

Filas con add_to_cart_order nulo: 0
Pedidos afectados: 0
Esta muestra no contiene pedidos con posicion de carrito nula.


**Interpretacion:** la fila del articulo contiene un pedido y un producto aunque falte la posicion en el carrito. Conservamos esos registros y describimos cuantos articulos hay en cada pedido afectado; la muestra reducida puede no incluir los 836 casos del conjunto completo.

### Ejercicio 4. Contar nombres cambiados por la estandarizacion

Carga de nuevo products.csv en products_crudo y cuenta cuantos nombres no nulos cambiaron despues de estandarizarlos.

In [67]:
products_crudo_reto = pd.read_csv(DATA_DIR / "products.csv", sep=";")
products_estandarizados_reto = limpiar_nombres(products_crudo_reto)
original_names = products_crudo_reto["product_name"]
standardized_names = products_estandarizados_reto["product_name"]
changed_non_null_names = original_names.notna() & original_names.ne(standardized_names)
changed_name_count = int(changed_non_null_names.sum())

print(f"Nombres no nulos que cambiaron: {changed_name_count:,}")
display(
    pd.DataFrame(
        {
            "original": original_names.loc[changed_non_null_names],
            "estandarizado": standardized_names.loc[changed_non_null_names],
        }
    ).head(10)
)

Nombres no nulos que cambiaron: 0


,original,estandarizado


**Interpretacion:** el conteo incluye solo nombres que existian antes de limpiar; los nulos originales, aunque se convierten en "Unknown", no se cuentan como nombres cambiados.

### Ejercicio 5. Comparar memoria de dia como texto y categoria ordenada

Crea una Series con el nombre del dia de cada pedido, conviertela a una categoria ordenada de domingo a sabado y compara la memoria utilizada.

In [68]:
weekday_names = ["domingo", "lunes", "martes", "mi\u00e9rcoles", "jueves", "viernes", "s\u00e1bado"]
weekday_text = orders_clean["order_dow"].map(
    dict(enumerate(weekday_names))
).astype("string")
weekday_category = pd.Series(
    pd.Categorical(weekday_text, categories=weekday_names, ordered=True),
    index=orders_clean.index,
    name="dia_categoria",
)

weekday_memory = pd.DataFrame(
    {
        "memoria_bytes": [
            int(weekday_text.memory_usage(deep=True)),
            int(weekday_category.memory_usage(deep=True)),
        ]
    },
    index=["texto", "categoria ordenada"],
)
display(weekday_memory)
display(pd.DataFrame({"texto": weekday_text, "categoria": weekday_category}).head())

,memoria_bytes
texto,353
categoria ordenada,857


,texto,categoria
0,domingo,domingo
1,lunes,lunes
2,domingo,domingo
3,martes,martes


**Interpretacion:** compara las dos cifras de memoria. Las categorias permiten ademas fijar el orden natural de los dias, independientemente de su orden alfabetico.

### Ejercicio 6. Revisar filas de productos Unknown

Cuantas filas de order_prods corresponden a productos cuyo product_name es "Unknown" y en cuantos pedidos aparecen? Decide si se deben eliminar.

In [65]:
order_prods_with_names = order_prods.merge(
    products_clean[["product_id", "product_name"]],
    on="product_id",
    how="left",
    validate="many_to_one",
)
unknown_items = order_prods_with_names.loc[
    order_prods_with_names["product_name"].eq("Unknown")
]
unknown_item_rows = len(unknown_items)
unknown_item_orders = int(unknown_items["order_id"].nunique())

print(f"Filas de productos Unknown: {unknown_item_rows:,}")
print(f"Pedidos con productos Unknown: {unknown_item_orders:,}")

Filas de productos Unknown: 0
Pedidos con productos Unknown: 0


**Interpretacion:** no se eliminan: son articulos que si aparecen en pedidos, aunque falte su nombre de catalogo. Quitarlos sesgaria los conteos y perderia referencias validas de compra.

### Ejercicio 7. Reemplazar missing en una copia de departments

En una copia de departments, reemplaza "missing" por pd.NA. Explica por que no conviene hacerlo en la tabla original antes de agrupar.

In [66]:
departments_for_analysis = departments_raw.copy()
departments_for_analysis["department"] = departments_for_analysis["department"].replace(
    "missing", pd.NA
)
display(departments_for_analysis)
print("La tabla original conserva sus valores:", departments_raw["department"].tolist())

,department_id,department
0,1,produce
1,2,dairy


La tabla original conserva sus valores: ['produce', 'dairy']


## 5. Conclusiones de la Leccion 2

- El pipeline reutilizable registra filas, nulos y memoria antes y despues de cada paso.
- Los tipos numericos quedan ajustados a su rango y las posiciones ausentes del carrito se conservan para investigar su causa.
- Los nombres de producto se estandarizan sin eliminar articulos ni confundir productos con ids distintos.
- Los siete ejercicios revisan duplicados, reglas de reordered, calidad de nombres, nulos en el carrito, estandarizacion, memoria categorica y valores "missing".
- Los resultados de estos CSV de ejemplo no sustituyen los resultados del conjunto completo de Instacart.

# Lección 3: filtrado, ordenamiento y agrupación de datos

Esta lección transforma las tablas limpias en respuestas sobre cuándo compran los clientes y con qué frecuencia vuelven. El laboratorio guiado construye un perfil de hábitos de compra por segmento de cliente; el Reto 3 resuelve los siete puntos del material oficial.

> Los archivos de este repositorio son muestras pequeñas. Los resultados sirven para comprobar el procedimiento, no para describir el conjunto completo de Instacart.

## 1. Laboratorio guiado 3: perfil de hábitos de compra

El área de mercadeo quiere diseñar campañas según la frecuencia de compra y conocer a qué hora compra cada tipo de cliente. La secuencia cuenta los pedidos por cliente, clasifica los clientes, asigna el segmento a cada pedido y cruza los segmentos con las franjas horarias. Trabajamos sobre copias de las tablas limpias para no alterar el resultado de la Lección 2.

### Paso 1. Contar los pedidos de cada cliente

Agrupamos por `user_id` y contamos los pedidos. La descripción estadística resume la distribución de pedidos por cliente.

In [69]:
import numpy as np

orders_analysis = orders_clean.copy()
order_products_analysis = order_products_clean.copy()

orders_per_customer = orders_analysis.groupby("user_id")["order_id"].count()
print(f"Clientes en la muestra: {len(orders_per_customer):,}")
display(orders_per_customer.describe().to_frame("pedidos_por_cliente"))

Clientes en la muestra: 3


,pedidos_por_cliente
count,3.000000
mean,1.333333
std,0.577350
min,1.000000
25%,1.000000
50%,1.000000
75%,1.500000
max,2.000000


### Paso 2. Segmentar clientes con `pd.cut`

Se definen tres grupos: ocasional (1 pedido), regular (2 a 4 pedidos) y frecuente (5 o más). Como `pd.cut` incluye por defecto el extremo derecho, los intervalos `(0, 1]`, `(1, 4]` y `(4, infinito)` representan esos grupos.

In [70]:
customer_segment = pd.cut(
    orders_per_customer,
    bins=[0, 1, 4, np.inf],
    labels=["ocasional", "regular", "frecuente"],
    ordered=True,
)

segment_counts = customer_segment.value_counts(sort=False).rename_axis("segmento")
segment_percentages = (
    customer_segment.value_counts(normalize=True, sort=False)
    .mul(100)
    .rename_axis("segmento")
)
segment_distribution = pd.DataFrame(
    {"clientes": segment_counts, "porcentaje": segment_percentages.round(1)}
)
display(segment_distribution)

,clientes,porcentaje
segmento,,
ocasional,2,66.7
regular,1,33.3
frecuente,0,0.0


### Paso 3. Asignar el segmento a cada pedido con `map`

El segmento está calculado por cliente; `map` lo lleva de vuelta al nivel de cada pedido. La columna se agrega a una copia para conservar `orders_clean` intacto.

In [71]:
orders_analysis["segmento"] = orders_analysis["user_id"].map(customer_segment)
unassigned_segments = int(orders_analysis["segmento"].isna().sum())
if unassigned_segments:
    raise ValueError(f"Hay {unassigned_segments} pedidos sin segmento de cliente.")

display(orders_analysis[["user_id", "order_number", "segmento"]].head())

,user_id,order_number,segmento
0,101,1,regular
1,102,1,ocasional
2,101,2,regular
3,103,1,ocasional


### Paso 4. Resumir por segmento

Calculamos clientes distintos, cantidad de pedidos y mediana de días hasta el pedido anterior. La mediana es menos sensible a valores extremos que la media; los valores nulos de la espera se omiten en su cálculo.

In [72]:
segment_summary = (
    orders_analysis.groupby("segmento", observed=True)
    .agg(
        clientes=("user_id", "nunique"),
        pedidos=("order_id", "count"),
        espera_mediana=("days_since_prior_order", "median"),
    )
)
display(segment_summary)

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,2,2,<NA>
regular,1,2,7.0


### Paso 5. Clasificar franjas horarias y cruzarlas con segmentos

Las franjas siguen los límites del material: madrugada (0–5), mañana (6–11), tarde (12–17) y noche (18–23). En `pd.cut`, los intervalos incluyen el extremo derecho; por eso el primer límite es `-1`, de modo que la hora 0 quede incluida.

In [73]:
time_bands = ["madrugada", "mañana", "tarde", "noche"]
orders_analysis["franja"] = pd.cut(
    orders_analysis["order_hour_of_day"],
    bins=[-1, 5, 11, 17, 23],
    labels=time_bands,
    ordered=True,
)

segment_time_profile = (
    pd.crosstab(
        orders_analysis["segmento"],
        orders_analysis["franja"],
        normalize="index",
    )
    .reindex(columns=time_bands, fill_value=0)
    .mul(100)
    .round(1)
)
display(segment_time_profile)

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,0,50.0,50.0,0
regular,0,100.0,0.0,0


### Paso 6. Encontrar las combinaciones de día y franja más activas

Agrupamos por día de la semana y franja, contamos los pedidos y ordenamos de mayor a menor para obtener las tres combinaciones más frecuentes.

In [74]:
top_day_time_combinations = (
    orders_analysis.groupby(["order_dow", "franja"], observed=True)
    .size()
    .sort_values(ascending=False)
    .head(3)
    .rename("pedidos")
    .reset_index()
)
display(top_day_time_combinations)

,order_dow,franja,pedidos
0,0,mañana,2
1,1,tarde,1
2,2,mañana,1


### Paso 7. Conservar las tablas de análisis sin columnas temporales

Las columnas `segmento` y `franja` solo son auxiliares del laboratorio. Se mantienen en `orders_analysis` para los retos, mientras que `orders_clean` queda exactamente como estaba al terminar la Lección 2.

In [75]:
assert "segmento" not in orders_clean.columns
assert "franja" not in orders_clean.columns
assert len(orders_analysis) == len(orders_clean)
print(f"orders_clean conserva su estructura: {orders_clean.shape}")
print(f"orders_analysis incluye los campos auxiliares: {orders_analysis.shape}")

orders_clean conserva su estructura: (4, 7)
orders_analysis incluye los campos auxiliares: (4, 9)


**Interpretación:** el perfil muestra cómo comparar horarios entre segmentos de distinto tamaño usando porcentajes por fila. Como estos CSV son muestras pequeñas, los conteos de esta ejecución no deben compararse con los valores de referencia del conjunto completo del curso.

## 2. Reto 3: filtrado, ordenamiento y agrupación

Para cada punto, expresamos la pregunta como una métrica por grupo o una condición y luego ejecutamos el código sobre las tablas limpias.

### Ejercicio 1. Tres horas con más pedidos los domingos

**Pregunta:** ¿cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?

In [76]:
sunday_orders = orders_analysis.loc[orders_analysis["order_dow"].eq(0)]
sunday_orders_by_hour = (
    sunday_orders.groupby("order_hour_of_day")["order_id"]
    .count()
    .sort_values(ascending=False)
    .head(3)
    .rename("pedidos")
    .rename_axis("hora")
    .reset_index()
)
display(sunday_orders_by_hour)
if sunday_orders_by_hour.empty:
    print("No hay pedidos del domingo en esta muestra.")

,hora,pedidos
0,10,2


**Interpretación:** la tabla está ordenada de mayor a menor frecuencia; el primer registro indica la hora con más pedidos dominicales de la muestra.

### Ejercicio 2. Horas con mayor y menor espera media

**Pregunta:** ¿a qué hora es mayor la espera media entre pedidos (`days_since_prior_order`) y a qué hora es menor? Se omiten horas cuya espera media sea nula.

In [77]:
mean_wait_by_hour = orders_analysis.groupby("order_hour_of_day")[
    "days_since_prior_order"
].mean().dropna()

if mean_wait_by_hour.empty:
    longest_wait_hours = pd.DataFrame(columns=["hora", "espera_media_dias"])
    shortest_wait_hours = pd.DataFrame(columns=["hora", "espera_media_dias"])
    print("No hay esperas previas válidas para calcular medias por hora.")
else:
    longest_wait_hours = mean_wait_by_hour.loc[
        mean_wait_by_hour.eq(mean_wait_by_hour.max())
    ].rename("espera_media_dias").rename_axis("hora").reset_index()
    shortest_wait_hours = mean_wait_by_hour.loc[
        mean_wait_by_hour.eq(mean_wait_by_hour.min())
    ].rename("espera_media_dias").rename_axis("hora").reset_index()

print("Mayor espera media:")
display(longest_wait_hours)
print("Menor espera media:")
display(shortest_wait_hours)

Mayor espera media:


,hora,espera_media_dias
0,10,7.0


Menor espera media:


,hora,espera_media_dias
0,10,7.0


**Interpretación:** se reportan todas las horas empatadas en el máximo o mínimo. La comparación describe asociación por hora en los datos, no demuestra que la hora cause una espera distinta.

### Ejercicio 3. Clientes con 10 pedidos o más

**Pregunta:** ¿cuántos clientes tienen 10 pedidos o más?

In [78]:
customers_with_10_or_more = int(orders_per_customer.ge(10).sum())
print(f"Clientes con 10 pedidos o más: {customers_with_10_or_more:,}")

Clientes con 10 pedidos o más: 0


### Ejercicio 4. Pedidos hechos exactamente 7 días después

**Pregunta:** ¿cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan? El denominador incluye todos los pedidos, también aquellos cuyo intervalo previo es nulo.

In [79]:
orders_after_7_days = int(orders_analysis["days_since_prior_order"].eq(7).sum())
orders_total = len(orders_analysis)
orders_after_7_days_pct = (
    orders_after_7_days / orders_total * 100 if orders_total else np.nan
)
print(f"Pedidos con intervalo de 7 días: {orders_after_7_days:,}")
print(f"Porcentaje del total de pedidos: {orders_after_7_days_pct:.2f}%")

Pedidos con intervalo de 7 días: 1
Porcentaje del total de pedidos: 25.00%


### Ejercicio 5. Pedidos con 20 productos o más

**Pregunta:** usando `order_prods`, ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?

In [80]:
products_per_order = order_products_analysis.groupby("order_id").size()
orders_with_20_or_more_items = int(products_per_order.ge(20).sum())
print(f"Pedidos con 20 productos o más: {orders_with_20_or_more_items:,}")

if products_per_order.empty:
    largest_orders = pd.DataFrame(columns=["order_id", "productos"])
    print("No hay filas de productos en los pedidos de esta muestra.")
else:
    largest_orders = products_per_order.loc[
        products_per_order.eq(products_per_order.max())
    ].rename("productos").rename_axis("order_id").reset_index()
    print(f"Tamaño máximo del pedido: {int(products_per_order.max())} productos")
    display(largest_orders)

Pedidos con 20 productos o más: 0
Tamaño máximo del pedido: 2 productos


,order_id,productos
0,1,2


### Ejercicio 6. Tasa de recompra por posición en el carrito

**Pregunta:** usando `order_prods`, calcula la media de `reordered` para las posiciones 1 a 5 del carrito. La media de una variable binaria es la proporción de artículos marcados como recompra.

In [81]:
cart_positions_1_to_5 = order_products_analysis.loc[
    order_products_analysis["add_to_cart_order"].between(1, 5)
]
reorder_rate_by_cart_position = (
    cart_positions_1_to_5.groupby("add_to_cart_order")["reordered"]
    .mean()
    .reindex(range(1, 6))
    .rename("tasa_recompra")
    .rename_axis("posicion_carrito")
    .reset_index()
)
display(reorder_rate_by_cart_position)

,posicion_carrito,tasa_recompra
0,1,0.333333
1,2,1.000000
2,3,NaN
3,4,NaN
4,5,NaN


**Interpretación:** compara las tasas entre posiciones; una posición sin artículos en la muestra aparecerá sin tasa (`NaN`). Describe el patrón observado sin asumir que el orden del carrito por sí mismo causa una recompra.

### Ejercicio 7. Comparar la espera de fin de semana con el resto de la semana

**Pregunta:** compara la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado, `order_dow` 0 y 6) con la del resto de la semana. Los intervalos ausentes se excluyen del cálculo de la mediana.

In [82]:
weekend_mask = orders_analysis["order_dow"].isin([0, 6])
weekend_wait_medians = pd.Series(
    {
        "fin_de_semana": orders_analysis.loc[
            weekend_mask, "days_since_prior_order"
        ].median(),
        "resto_de_la_semana": orders_analysis.loc[
            ~weekend_mask, "days_since_prior_order"
        ].median(),
    },
    name="mediana_dias_entre_pedidos",
).to_frame()
display(weekend_wait_medians)

,mediana_dias_entre_pedidos
fin_de_semana,7.0
resto_de_la_semana,<NA>


**Interpretación:** compara ambas medianas cuando estén disponibles. Una mediana ausente indica que en ese grupo no hay intervalos previos válidos en la muestra; no debe sustituirse por cero.

## 3. Conclusiones de la Lección 3

- El filtrado selecciona los pedidos que cumplen condiciones explícitas; `groupby` resume métricas por cliente, hora, segmento o posición del carrito.
- `pd.cut` convierte conteos y horas en categorías con límites definidos, y `map` traslada al detalle los segmentos calculados por cliente.
- Las proporciones por fila de la tabla cruzada permiten comparar segmentos de distinto tamaño.
- Los valores ausentes de `days_since_prior_order` se excluyen de la media o mediana de forma intencional, ya que corresponden a pedidos sin intervalo previo registrado.
- Los resultados de esta ejecución describen únicamente los CSV de muestra del repositorio.

# Lecci?n 4: combinaci?n de datasets y tabla maestra

Esta lecci?n integra las tablas limpias para construir una vista unificada de cada producto en cada pedido. El laboratorio guiado crea una tabla maestra confiable; el Reto 4 usa la misma estructura para responder preguntas de negocio sobre categor?as, pedidos y recompra.

> Los resultados de esta muestra son descriptivos del conjunto disponible en el repositorio. El objetivo principal es comprobar el procedimiento de merge y la integridad de la informaci?n combinada.


## 1. Laboratorio guiado 4: combinaci?n de datasets

### Paso 1. Preparar la ruta de salida y reutilizar los datos de an?lisis


In [ ]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(exist_ok=True, parents=True)
orders_for_merge = orders_analysis.copy()
print(f"Directorio de salida: {processed_dir.resolve()}")


### Paso 2. Unir productos, pasillos, departamentos y pedidos


In [ ]:
order_items = order_products_analysis.merge(
    products_clean[["product_id", "product_name", "aisle_id", "department_id"]],
    on="product_id",
    how="left",
    validate="many_to_one",
)

order_items = order_items.merge(
    aisles_clean[["aisle_id", "aisle"]],
    on="aisle_id",
    how="left",
    validate="many_to_one",
)

master_table = (
    order_items.merge(
        departments_clean[["department_id", "department"]],
        on="department_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        orders_for_merge[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]],
        on="order_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values(["order_id", "add_to_cart_order"])
    .reset_index(drop=True)
)

assert set(["order_id", "product_id", "department", "aisle"]).issubset(master_table.columns)
print(f"Filas en la tabla maestra: {len(master_table):,}")
print(master_table.head(8).to_string(index=False))


### Paso 3. Resumir la combinaci?n por departamento y producto


In [ ]:
department_summary = (
    master_table.groupby("department", dropna=False)
    .agg(
        pedidos=("order_id", "nunique"),
        productos_unicos=("product_id", "nunique"),
        unidades=("product_id", "count"),
        tasa_recompra=("reordered", "mean"),
    )
    .sort_values(["unidades", "pedidos"], ascending=False)
    .reset_index()
)

print("Resumen por departamento:")
print(department_summary.to_string(index=False))


### Paso 4. Guardar versiones procesadas en data/processed


In [ ]:
master_table.to_csv(processed_dir / "instacart_master_table.csv", index=False)

department_summary.to_csv(processed_dir / "department_summary.csv", index=False)

assert (processed_dir / "instacart_master_table.csv").exists()
assert (processed_dir / "department_summary.csv").exists()
print("Archivos generados:")
for path in sorted(processed_dir.iterdir()):
    print(path.name)


## 2. Reto 4: siete ejercicios de combinaci?n de datasets

### Ejercicio 1. ?Cu?ntas filas contiene la tabla maestra?


In [ ]:
total_master_rows = int(len(master_table))
print(f"Filas totales en la tabla maestra: {total_master_rows:,}")


### Ejercicio 2. ?Cu?ntos registros carecen de pasillo o departamento?


In [ ]:
missing_context = int(master_table[["aisle", "department"]].isna().any(axis=1).sum())
print(f"Registros sin contexto de pasillo o departamento: {missing_context:,}")


### Ejercicio 3. ?Cu?l es la tasa media de recompra por departamento?


In [ ]:
reorder_rate_by_department = (
    master_table.groupby("department", dropna=False)["reordered"]
    .mean()
    .sort_values(ascending=False)
    .rename("tasa_recompra")
    .reset_index()
)
print(reorder_rate_by_department.to_string(index=False))


### Ejercicio 4. ?Qu? cinco departamentos tienen m?s pedidos?


In [ ]:
top_departments_by_orders = (
    master_table.groupby("department", dropna=False)["order_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(5)
    .rename("pedidos_unicos")
    .reset_index()
)
print(top_departments_by_orders.to_string(index=False))


### Ejercicio 5. ?Cu?ntos productos distintos compra cada usuario?


In [ ]:
unique_products_by_user = (
    master_table.groupby("user_id", dropna=False)["product_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("productos_distintos")
    .reset_index()
)
print(unique_products_by_user.to_string(index=False))


### Ejercicio 6. ?Cu?l es la media de productos por pedido seg?n el d?a de la semana?


In [ ]:
items_per_order_by_weekday = (
    master_table.groupby("order_dow")
    .agg(productos_totales=("product_id", "count"), pedidos_unicos=("order_id", "nunique"))
    .assign(productos_por_pedido=lambda x: x["productos_totales"] / x["pedidos_unicos"])    .sort_values("productos_por_pedido", ascending=False)
)
print(items_per_order_by_weekday.to_string())


### Ejercicio 7. Validar la persistencia de los archivos procesados


In [ ]:
processed_files = sorted(path.name for path in processed_dir.iterdir())
print(f"Archivos generados: {processed_files}")
assert "instacart_master_table.csv" in processed_files
assert "department_summary.csv" in processed_files


## 3. Conclusiones de la Lecci?n 4

- La combinaci?n de tablas por clave primaria y for?nea permite construir una vista unificada del pedido con contexto de producto, pasillo, departamento y cliente.
- La validaci?n con `validate="many_to_one"` ayuda a detectar errores de cardinalidad antes de seguir con an?lisis m?s complejos.
- El guardado en `data/processed/` crea un dataset reutilizable para futuras exploraciones, informes o modelos.
- La tabla maestra mantiene el nivel de detalle de los items del pedido y facilita res?menes por usuario, departamento y frecuencia de recompra.
